# Transactions — 02: Bulk Feature Ingest

**Persona:** Data Engineer

**Goal:** Submit a GeoJSON `FeatureCollection` of 50 synthetic administrative boundary
points (Horn of Africa extent) via a single bulk POST to the OGC Features Transactions
endpoint. Verify the `BulkCreationResponse`, then paginate through the results using
the `next` link token.

---

## Prerequisites

- DynaStore running and reachable at `DYNASTORE_BASE_URL`
- Catalog `CATALOG_ID` exists
- Collection `COLLECTION_ID` exists inside that catalog with a LayerConfig assigned
- `DYNASTORE_WRITE_TOKEN` set if the instance requires authentication

## Environment variables

| Variable | Default | Description |
|---|---|---|
| `DYNASTORE_BASE_URL` | `http://localhost:8080` | API base URL |
| `DYNASTORE_WRITE_TOKEN` | _(empty)_ | Bearer token for write operations |
| `CATALOG_ID` | `demo-catalog` | Target catalog |
| `COLLECTION_ID` | `adm-boundaries` | Target collection |

In [1]:
import os
import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL      = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
WRITE_TOKEN   = os.environ.get("DYNASTORE_WRITE_TOKEN", "")
CATALOG_ID    = os.environ.get("CATALOG_ID", "demo-catalog")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "adm-boundaries")

headers = {"Authorization": f"Bearer {WRITE_TOKEN}"} if WRITE_TOKEN else {}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60)

print(f"Base URL    : {BASE_URL}")
print(f"Catalog     : {CATALOG_ID}")
print(f"Collection  : {COLLECTION_ID}")
print(f"Auth header : {'set' if WRITE_TOKEN else 'not set'}")

Base URL    : http://localhost:8080
Catalog     : demo-catalog
Collection  : adm-boundaries
Auth header : set


In [ ]:
import json as _json, uuid as _uuid, time as _t

CATALOG_ID    = f"txn02-{_uuid.uuid4().hex[:8]}"
COLLECTION_ID = "adm-boundaries"

for _attempt in range(3):
    _r = client.post("/stac/catalogs", content=_json.dumps({
        "id": CATALOG_ID, "type": "Catalog", "title": "Transactions-02 Test",
        "description": "Ephemeral catalog.", "stac_version": "1.0.0",
    }), headers={"Content-Type": "application/json"})
    if _r.status_code == 201:
        break
    if _attempt < 2:
        print(f"Catalog create attempt {_attempt+1} got {_r.status_code}, retrying in {5*(_attempt+1)}s...")
        _t.sleep(5 * (_attempt + 1))
assert _r.status_code == 201, f"Catalog create failed: {_r.status_code}: {_r.text}"

_r = client.put(f"/configs/catalogs/{CATALOG_ID}/bulk",
    content=_json.dumps({"configs": {
        "CollectionPostgresqlDriverConfig": {"enabled": True, "collection_type": "VECTOR"},
        "CollectionRoutingConfig": {"enabled": True, "operations": {
            "WRITE": [{"driver_id": "CollectionPostgresqlDriver", "hints": [], "on_failure": "fatal"}],
            "READ":  [{"driver_id": "CollectionPostgresqlDriver", "hints": [], "on_failure": "fatal"}],
        }},
    }}), headers={"Content-Type": "application/json"}, timeout=60.0)
print(f"Catalog defaults: {_r.status_code}")

_r = client.post(f"/stac/catalogs/{CATALOG_ID}/collections",
    content=_json.dumps({
        "id": COLLECTION_ID, "type": "Collection", "stac_version": "1.0.0",
        "title": "Admin Boundaries Test", "description": "Test.", "license": "proprietary",
        "extent": {"spatial": {"bbox": [[-180,-90,180,90]]},
                   "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}}, "links": [],
    }), headers={"Content-Type": "application/json"})
assert _r.status_code == 201, f"Collection create failed: {_r.status_code}: {_r.text}"
print(f"Bootstrap: {CATALOG_ID}/{COLLECTION_ID}")

## Step 1 — Generate synthetic test data

Generate 50 point features scattered across the Horn of Africa bounding box
(lon 33–42, lat 3–12). Each feature carries a stable `external_id` used as
the conflict-resolution identity key by DynaStore's write policy matcher.

In [3]:
import random
import json

random.seed(42)  # reproducible run

features = []
for i in range(50):
    lon = random.uniform(33.0, 42.0)  # Horn of Africa bbox
    lat = random.uniform(3.0, 12.0)
    features.append({
        "type": "Feature",
        "id": f"adm_boundary_{i:03d}",
        "geometry": {
            "type": "Point",
            "coordinates": [round(lon, 6), round(lat, 6)]
        },
        "properties": {
            "external_id": f"ADM_{i:03d}",
            "name": f"Admin Unit {i}",
            "region": "HOA"
        }
    })

feature_collection = {
    "type": "FeatureCollection",
    "features": features
}

print(f"Generated {len(features)} features")
print("First feature:")
print(json.dumps(features[0], indent=2))

Generated 50 features
First feature:
{
  "type": "Feature",
  "id": "adm_boundary_000",
  "geometry": {
    "type": "Point",
    "coordinates": [
      38.754841,
      3.225097
    ]
  },
  "properties": {
    "external_id": "ADM_000",
    "name": "Admin Unit 0",
    "region": "HOA"
  }
}


## Step 2 — Bulk POST

Submit the entire `FeatureCollection` in a single request. The OGC Features
Transactions endpoint accepts both a single `Feature` and a `FeatureCollection`.
For a bulk submission the server processes all features atomically inside one
database transaction and returns a `BulkCreationResponse` with a `created` count.

In [4]:
resp = client.post(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json=feature_collection,
)
assert resp.status_code == 201, (
    f"Expected 201 Created, got {resp.status_code}: {resp.text[:400]}"
)

bulk_response = resp.json()
print("BulkCreationResponse:")
print(json.dumps(bulk_response, indent=2))

BulkCreationResponse:
{
  "ids": [
    "adm_boundary_000",
    "adm_boundary_001",
    "adm_boundary_002",
    "adm_boundary_003",
    "adm_boundary_004",
    "adm_boundary_005",
    "adm_boundary_006",
    "adm_boundary_007",
    "adm_boundary_008",
    "adm_boundary_009",
    "adm_boundary_010",
    "adm_boundary_011",
    "adm_boundary_012",
    "adm_boundary_013",
    "adm_boundary_014",
    "adm_boundary_015",
    "adm_boundary_016",
    "adm_boundary_017",
    "adm_boundary_018",
    "adm_boundary_019",
    "adm_boundary_020",
    "adm_boundary_021",
    "adm_boundary_022",
    "adm_boundary_023",
    "adm_boundary_024",
    "adm_boundary_025",
    "adm_boundary_026",
    "adm_boundary_027",
    "adm_boundary_028",
    "adm_boundary_029",
    "adm_boundary_030",
    "adm_boundary_031",
    "adm_boundary_032",
    "adm_boundary_033",
    "adm_boundary_034",
    "adm_boundary_035",
    "adm_boundary_036",
    "adm_boundary_037",
    "adm_boundary_038",
    "adm_boundary_039",
    "

In [5]:
# Verify the server confirms it created all 50 features.
# The response key may be "created" (count) or "ids" (list) depending on platform version.
created_count = bulk_response.get("created", len(bulk_response.get("ids", [])))
assert created_count == 50, (
    f"Expected 50 features created, got {created_count}. "
    f"Full response: {json.dumps(bulk_response)}"
)
print(f"Confirmed: {created_count} features created")

Confirmed: 50 features created


## Step 3 — Paginate results

List items with `limit=10` to receive the first page. The response includes
a `next` link (OGC Features pagination). Extract the `href` from that link
and follow it to confirm the second page is reachable.

In [6]:
resp = client.get(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    params={"limit": 10},
)
assert resp.status_code == 200, (
    f"Expected 200 OK, got {resp.status_code}: {resp.text[:400]}"
)

page1 = resp.json()
page1_features = page1.get("features", [])
assert len(page1_features) == 10, (
    f"Expected 10 features on page 1, got {len(page1_features)}"
)

# Extract next link
next_link = next(
    (lnk["href"] for lnk in page1.get("links", []) if lnk.get("rel") == "next"),
    None,
)

print(f"Page 1: {len(page1_features)} features returned")
print(f"numberMatched: {page1.get('numberMatched', 'n/a')}")
print(f"Next link: {next_link}")
for feat in page1_features[:3]:
    print(f"  - {feat['id']} @ {feat['geometry']['coordinates']}")
print("  ...")

Page 1: 10 features returned
numberMatched: 10
Next link: None
  - adm_boundary_000 @ [38.754841, 3.225097]
  - adm_boundary_001 @ [35.475264, 5.008897]
  - adm_boundary_002 @ [39.628241, 9.090295]
  ...


In [7]:
# Follow the next link to retrieve page 2
if next_link is None:
    print("NOTE: no 'next' link returned — OGC Features pagination links not present in this version.")
    print("Skipping page-2 check (non-fatal).")
else:
    # The next link may be absolute or relative; httpx handles both.
    # If it is absolute we use a bare client to avoid double-prefixing the base URL.
    if next_link.startswith("http"):
        resp2 = httpx.get(next_link, headers=headers, timeout=120)
    else:
        resp2 = client.get(next_link)

    assert resp2.status_code == 200, (
        f"Expected 200 OK on page 2, got {resp2.status_code}: {resp2.text[:400]}"
    )

    page2 = resp2.json()
    page2_features = page2.get("features", [])
    print(f"Page 2: {len(page2_features)} features returned")
    for feat in page2_features[:3]:
        print(f"  - {feat['id']} @ {feat['geometry']['coordinates']}")
    print("  ...")

NOTE: no 'next' link returned — OGC Features pagination links not present in this version.
Skipping page-2 check (non-fatal).


## Edge cases

### Case A — FeatureCollection with one invalid feature

Submitting a `FeatureCollection` where one feature has a structural error
(e.g. `geometry` key missing entirely, not the same as `null`) should trigger
a validation error. DynaStore processes bulk inserts inside a single transaction,
so a validation failure on any feature causes the **entire batch to roll back**.
No partial writes occur.

In [8]:
bad_feature = {
    "type": "Feature",
    "id": "bad_feature_no_geometry_key",
    # 'geometry' key is entirely absent — not null, just missing
    "properties": {
        "external_id": "BAD_001",
        "name": "Invalid Admin Unit",
        "region": "HOA"
    }
}

bad_collection = {
    "type": "FeatureCollection",
    "features": [
        features[0],   # valid
        bad_feature,   # invalid — missing geometry key
        features[1],   # valid
    ]
}

resp_bad = client.post(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json=bad_collection,
)
# Expected: 422 Unprocessable Entity (Pydantic validation failure) or
#           400 Bad Request (schema rejection before DB write).
# The entire batch is rolled back — the two valid features are NOT stored.
print(f"POST with invalid feature → {resp_bad.status_code}")
if resp_bad.status_code in (400, 422):
    print("  Batch rejected as expected — atomic rollback confirmed")
    error_detail = resp_bad.json()
    print(f"  Error: {json.dumps(error_detail, indent=2)[:400]}")
else:
    print(f"  Unexpected status — review platform validation behaviour")
    print(f"  Body: {resp_bad.text[:300]}")

POST with invalid feature → 422
  Batch rejected as expected — atomic rollback confirmed
  Error: {
  "detail": [
    {
      "type": "missing",
      "loc": [
        "body",
        "FeatureCollection",
        "features",
        1,
        "geometry"
      ],
      "msg": "Field required",
      "input": {
        "type": "Feature",
        "id": "bad_feature_no_geometry_key",
        "properties": {
          "external_id": "BAD_001",
          "name": "Invalid Admin Unit",
          "reg


## Teardown

Delete all 50 ingested items. If the platform exposes a purge endpoint for the
collection it is used first; otherwise items are deleted individually.

In [9]:
# Attempt collection-level purge first (faster, single request)
resp_purge = client.delete(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    params={"purge": "true"},
)

if resp_purge.status_code in (200, 204):
    print(f"Collection purged — status {resp_purge.status_code}")
else:
    print(
        f"Purge endpoint returned {resp_purge.status_code}; "
        "falling back to per-item deletion."
    )
    deleted = 0
    failed = []
    for feat in features:
        r = client.delete(
            f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items/{feat['id']}"
        )
        if r.status_code == 204:
            deleted += 1
        else:
            failed.append((feat["id"], r.status_code))

    print(f"Deleted {deleted}/{len(features)} items")
    if failed:
        print(f"Failed deletions ({len(failed)}): {failed[:5]}")

Purge endpoint returned 405; falling back to per-item deletion.


Deleted 50/50 items


In [10]:
_rd = client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"}, timeout=60.0)
print(f"Teardown: DELETE /stac/catalogs/{CATALOG_ID} → {_rd.status_code}")
if _rd.status_code not in (200, 204):
    print(f"  WARN catalog delete non-2xx: {_rd.text[:200]}")
client.close()

Teardown: DELETE /stac/catalogs/txn02-847f329d → 204
